# 计算引擎

引擎负责执行完整的计算图。日常使用主要需要选择引擎、传入目标和请求、处理失败；各引擎统一使用 `run()` 入口，结果顺序与目标列表一致。

遍历顺序、进程通信和同步机制集中在[原理和细节](原理和细节.ipynb)。

## 先选哪一个引擎

| 需求 | 引擎 | 使用条件 |
|---|---|---|
| 开发调试、一般顺序计算 | `Engine` | 优先从它开始验证结果 |
| 需要栈式遍历的自定义流程 | `StackEngine` | 沿用相同调用接口，注意生命周期顺序差异 |
| 同一任务按资产等数据分片并行 | `ParallelEngine` | 上下文和请求支持切分，目标节点实现结果合并 |
| 不同依赖分支可以同时计算 | `TreeEngine` | 节点适合并发，线程共享资源或进程同步均需匹配业务实现 |

切换引擎不会自动使自定义节点具备数据切分、缓存或线程安全能力。先用 `Engine` 检查正确性，再在有实际耗时的任务上比较并行收益。

## 最小运行示例

下面将计算配置放入参数模型，使用两个独立目标演示返回顺序。节点实现方式见[计算图框架](计算图框架.ipynb)。

In [1]:
import logging
from QuantStudio.Core import setDefaultLogLevel

setDefaultLogLevel(logging.WARNING)

2026-10-31 11:31:56,971 | QS | 29492 | ERROR : 节点 错误输入(eba7c216) 计算失败: 示例输入无效
2026-10-31 11:31:56,972 | QS | 29492 | WARNING : 计算图执行完成, 共 1 个节点失败: 错误输入(981e9ae3)


In [2]:
from pydantic import Field
from QuantStudio.Core.api import Node, DependencyModel, Context, Engine


class Number(Node):
    """返回参数中声明的数值。"""

    class __QS_DepClass__(DependencyModel):
        """数字节点没有依赖。"""

    class __QS_ArgClass__(Node.__QS_ArgClass__):
        Value: float = Field(frozen=True, title="数值")

    def backward_compute(self, path, inputs, context, local_context=None):
        """返回数值。

        Args:
            path: 当前运行路径。
            inputs: 无依赖的结果视图。
            context: 本次运行上下文。
            local_context: 局部上下文。

        Returns:
            参数声明的数值。
        """
        return self.Args.Value


class Add(Node):
    """将首项与其余各项相加。"""

    class __QS_DepClass__(DependencyModel):
        first: Node = Field(title="首项")
        others: tuple[Node, ...] = Field(min_length=1, title="其他项")

    def backward_compute(self, path, inputs, context, local_context=None):
        """汇总依赖结果。

        Args:
            path: 当前运行路径。
            inputs: 按 first、others 组织的依赖结果。
            context: 本次运行上下文。
            local_context: 局部上下文。

        Returns:
            所有依赖结果的和。
        """
        return inputs.first + sum(inputs.others)

In [3]:
left = Add(deps={"first": Number(args={"Value": 1}),
                 "others": (Number(args={"Value": 2}),)}, args={"Name": "一加二"})
right = Add(deps={"first": Number(args={"Value": 3}),
                  "others": (Number(args={"Value": 4}),)}, args={"Name": "三加四"})
targets = [left, right]

engine = Engine()
results = engine.run(targets, Context())
assert results == [3, 7]
print(results)  # [3.0, 7.0]

[3.0, 7.0]


## 调用接口与常用参数

```python
results = engine.run(
    node_list,
    context,
    fwd_data_list=None,
    init_data_list=None,
)
```

- `node_list`：完整的目标节点，不能放入模板或算子。引擎自动处理依赖。
- `context`：本次任务的全局上下文；业务模块通常使用自己的子类。
- `fwd_data_list`：各目标的计算请求，数量应与目标一致，建议使用关键字传参。
- `init_data_list`：各目标的初始化请求；省略时使用 `fwd_data_list`。

| 参数 | 默认值 | 用法 |
|---|---|---|
| `IOConcurrentNum` | `None` | 准备阶段的线程并发数；`None` 按登记的准备任务数确定，`1` 顺序准备 |
| `FailMode` | `"raise"` | 节点失败时抛出异常；设为 `"skip"` 可继续处理不受影响的目标 |

参数在构造引擎时通过 `args` 传入。IO 并发只作用于准备阶段，不会把 `Engine` 的主计算变成并行计算。数据源连接和自定义准备方法需要支持相应的并发访问。

In [4]:
engine = Engine(args={"IOConcurrentNum": 1, "FailMode": "raise"})
assert engine.run(targets, Context()) == [3, 7]

## StackEngine：使用相同接口进行栈式计算

`StackEngine` 通过显式栈组织初始化和前向、后向计算。普通任务可以继续使用 `Engine`；需要此遍历方式时，替换引擎对象即可。其内部顺序见[原理和细节](原理和细节.ipynb)。

In [5]:
from QuantStudio.Core.CalcEngine import StackEngine

results = StackEngine().run(targets, Context())
assert results == [3, 7]
print(results)

[3.0, 7.0]


## TreeEngine：让独立分支并发计算

使用 `CalcConcurrentNum` 指定计算并发数，使用 `CalcConcurrentMode` 选择 `"Thread"` 或 `"Process"`（默认）。

| 模式 | 适用情况 | 使用要求 |
|---|---|---|
| `"Thread"` | IO 等待较多，或主要计算在释放 GIL 的库中完成 | 各节点共享上下文，避免无同步地修改共享对象 |
| `"Process"` | CPU 计算较多，足以抵消进程通信开销 | 节点、请求及资源适合跨进程传递，需要保留的变更由具体上下文同步 |

下面使用线程模式，便于直接在 notebook 中执行。返回结果仍按目标顺序排列；两个任务完成的先后顺序不保证。这个小例子用于说明接口，并行未必更快。

In [6]:
from QuantStudio.Core.api import TreeEngine

tree_engine = TreeEngine(args={"CalcConcurrentMode": "Thread", "CalcConcurrentNum": 2})
results = tree_engine.run(targets, Context())
assert results == [3, 7]
print(results)

[3.0, 7.0]


## ParallelEngine：按数据分片并行

`ParallelEngine` 将上下文和请求分成多个任务，每个任务执行目标图，再调用目标节点的 `merge_result()` 合并结果。进程数量由 `context.PIDList` 决定；请求、上下文和节点的分片策略必须配套。

对于因子任务，优先使用模块已有的 `FactorContext`、`FactorLocalContext` 和缓存实现。下面使用固定的本地数据，将两个资产分给两个进程；无需外部数据库。普通标量节点不适合仅靠设置 `PIDList` 来加速。

在 Windows 独立脚本中启动多进程时，将运行入口放入 `if __name__ == "__main__":` 中。

In [7]:
import datetime as dt
import pandas as pd
from QuantStudio.Core.api import ParallelEngine
from QuantStudio.Factor.Factor import DataFactor, FactorContext, FactorLocalContext

dates = [dt.datetime(2025, 1, day) for day in range(1, 4)]
ids = ["A", "B"]
data = pd.DataFrame([[1., 2.], [3., 4.], [5., 6.]], index=dates, columns=ids)
factor = DataFactor(data)
parallel_context = FactorContext(DTRuler=dates, SectionIDs=ids, PIDList=["0", "1"])
request = FactorLocalContext(DTs=dates, IDs=ids)

results = ParallelEngine().run([factor], parallel_context, fwd_data_list=[request])
pd.testing.assert_frame_equal(results[0], data, check_freq=False)
print(results[0])

  0% (0 of 2) |                          | Elapsed Time: 0:00:00 ETA:  --:--:--
100% (2 of 2) |##########################| Elapsed Time: 0:00:00 ETA:  00:00:00
100% (2 of 2) |##########################| Elapsed Time: 0:00:00 Time:  0:00:00


              A    B
2025-01-01  1.0  2.0
2025-01-02  3.0  4.0
2025-01-03  5.0  6.0


## 部分失败时继续运行

默认 `FailMode="raise"` 会将异常抛给调用方。允许部分目标失败时，可以使用 `"skip"`：失败目标及依赖它的目标返回 `None`，不受影响的目标仍可完成。

检查 `context.NodeState` 的 `__status__` 判断失败或跳过，检查 `context.NodeErrors` 获取原始异常。不要仅以结果是否为 `None` 判断，因为自定义节点也可能正常返回 `None`。

In [8]:
class InvalidNumber(Number):
    """演示一个会失败的节点。"""

    def backward_compute(self, path, inputs, context, local_context=None):
        """抛出示例异常。"""
        raise ValueError("示例输入无效")


invalid = InvalidNumber(args={"Value": 0, "Name": "错误输入"})
affected = Add(deps={"first": invalid, "others": (Number(args={"Value": 1}),)},
               args={"Name": "受影响的目标"})
context = Context()
results = Engine(args={"FailMode": "skip"}).run([affected, right], context)
assert results == [None, 7]

for target, result in zip([affected, right], results):
    status = context.NodeState.get(target.InstanceID, {}).get("__status__")
    print(target.Name, status or "已完成", result)
for instance_id, error in context.NodeErrors.items():
    print(context.NodeDict[instance_id].Name, str(error))

受影响的目标 SKIPPED None
三加四 已完成 7.0
错误输入 示例输入无效


## 获取执行进度

将可调用对象赋给 `Context.EventHandler`，可接收包含 `stage`、`status`、`run_id`、`instance_id`、`qsid` 等字段的事件。回调应保持轻量，并自行处理展示方式。各引擎的事件顺序和数量可能不同，不应将某一种固定顺序作为业务计算的前提。

In [9]:
events = []
context = Context(EventHandler=events.append)
assert Engine().run([left], context) == [3]
assert events and all(event["run_id"] == context.RunID for event in events)
print(sorted({(event["stage"], event["status"]) for event in events}))

[('compute', 'done'), ('compute', 'running'), ('init', 'done'), ('init', 'running')]


## 在业务接口中临时切换引擎

显式调用 `engine.run()` 时直接使用该对象即可。部分因子、回测便捷接口会读取当前活动引擎，此时可以用 `with` 在一段代码内切换，退出后恢复。

```python
with TreeEngine(args={"CalcConcurrentMode": "Thread", "CalcConcurrentNum": 2}):
    # 调用读取当前活动引擎的业务接口，例如 factor.readData(...)
    ...
```

进入 `with` 本身不会执行计算。全局引擎栈、不同引擎的生命周期顺序与自定义同步要求见[原理和细节](原理和细节.ipynb)。